In [1]:
import requests
import html2text
import re
from bs4 import BeautifulSoup

def download_webpage_enhanced(url, output_file=None):
    """
    增强版网页下载和转换
    
    特点:
    1. 更好的HTML清理
    2. 更多的格式控制
    3. 可选的BeautifulSoup预处理
    """
    
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # 可选：使用BeautifulSoup预处理
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 移除脚本、样式、注释等
        for element in soup(["script", "style", "nav", "footer", "header"]):
            element.decompose()
        
        html_content = str(soup)
        
    except Exception as e:
        print(f"下载失败: {e}")
        return None
    
    # 配置转换器
    h = html2text.HTML2Text()
    
    # 配置选项
    h.ignore_links = True
    h.ignore_images = True
    h.ignore_emphasis = True
    h.skip_internal_links = True
    h.inline_links = False
    h.protect_links = False
    h.body_width = 0
    h.unicode_snob = True
    h.wrap_links = False
    
    # 转换
    text = h.handle(html_content)
    
    # 查找并删除"Status"之前的内容
    # 使用正则表达式，不区分大小写
    pattern = re.compile(r'^.*?(?=status)', re.IGNORECASE | re.DOTALL)
    text = pattern.sub('', text, count=1)
    
    # 清理文本
    text = clean_text(text)
    
    if output_file:
        save_text(text, output_file)
    
    return text

def clean_text(text):
    """清理文本的辅助函数"""
    # 移除URL
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    
    # 移除多余空白
    text = re.sub(r'[ \t]+', ' ', text)  # 多个空格/制表符替换为一个空格
    text = re.sub(r'\n{3,}', '\n\n', text)  # 多个空行替换为两个空行
    
    # 移除行首行尾空白
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    
    return text.strip()

def save_text(text, filename):
    """保存文本到文件"""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(text)
    #print(f"文件已保存: {filename}")

In [9]:
###先下载所有的
import pandas as pd
dt = pd.read_csv("curr_gsm.txt",sep="\t")

In [10]:
dt

,experiment_geo_accession,srrs,organ,pair
0,GSM3358239,SRR7760431,Mus_musculus,SINGLE
1,GSM3358240,SRR7760432,Mus_musculus,SINGLE
2,GSM3358241,SRR7760433,Mus_musculus,SINGLE
3,GSM3358242,SRR7760434,Mus_musculus,SINGLE
4,GSM3358243,SRR7760435,Mus_musculus,SINGLE
...,...,...,...,...
129,GSM8226813,SRR28785554,Mus_musculus,PAIRED
130,GSM8226814,SRR28785553,Mus_musculus,PAIRED
131,GSM8226815,SRR28785552,Mus_musculus,PAIRED
132,GSM8226816,SRR28785551,Mus_musculus,PAIRED


In [4]:
from tqdm import tqdm

In [11]:
import time
for i in tqdm(range(len(dt))):
    text = download_webpage_enhanced("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc="+dt.experiment_geo_accession[i],
                                     "GSM_text/"+dt.experiment_geo_accession[i]+".txt")
    if text is None:
        print("无数据，等待30秒...")
        time.sleep(30)
        text = download_webpage_enhanced("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc="+dt.experiment_geo_accession[i],
                                         "GSM_text/"+dt.experiment_geo_accession[i]+".txt") 
        if text is None:
            time.sleep(30)
            text = download_webpage_enhanced("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc="+dt.experiment_geo_accession[i],
                                             "GSM_text/"+dt.experiment_geo_accession[i]+".txt") 
            if text is None:
                continue

100%|██████████| 134/134 [02:14<00:00,  1.00s/it]


In [13]:
prompts = """Based on the following text, extract the name of the cell line, the drug treatments on the cell line, and the concentration and duration of the drug treatments, do not provide extra explanations:
{text}

Output Format:
Cell Line: [cell line name]
Treatment: [drug name]
Treatment concentration: [concentration]
Treatment time: [duration of the drug treatment]
"""

In [12]:
import os
from openai import OpenAI
import pandas as pd
dt = pd.read_csv("curr_gsm.txt")

client = OpenAI(
    api_key="empt",
    base_url="http://10.19.28.74:11434/v1"
)

In [14]:
dt

,gsms
0,GSM1519353
1,GSM1519354
2,GSM1519355
3,GSM1519356
4,GSM1519357
...,...
2206,GSM8729249
2207,GSM8729250
2208,GSM8729251
2209,GSM8729252


In [19]:
from tqdm import tqdm
for i in tqdm(range(len(dt))):
    with open('GSM_text/'+dt.gsms[i]+".txt", 'r', encoding='utf-8') as file:
        text = file.read()  # 读取整个文件内容
    response = client.chat.completions.create(
        model="alibayram/Qwen3-30B-A3B-Instruct-2507:latest",
        messages=[
            {"role": "system", "content": "You are an expert in biology and need to extract information from text."},
            {"role": "user", "content": prompts.format(text = text)},
        ],
        stream=False)
    with open('GSM_deepseek/'+dt.gsms[i]+"_out.txt", 'w', encoding='utf-8') as f:
        print(response.choices[0].message.content, file=f)

100%|██████████| 2211/2211 [27:58<00:00,  1.32it/s]


In [24]:
prompts = """Based on the following text, check whether the following information is correct, Check for each of the four field:

Cell Line: {cell}
Treatment: {drug}
Treatment concentration: {drug_con}
Treatment time: {drug_time}

Cell Line refer to the name of Cell Line.
Treatment refer to the name of the drug or agent treated on the cell line. In particular, check whether the treatment is DMSO or untreated (pay more attention in Title).
Treatment concentration refer to the concentration of the drug or the agent.
Treatment time refer to the time of treatment.

If it is correct, output as the same, if not correct, correct the error information, do not provide extra explanations. The output format should same as input.
If has multiple treatments, only keep the frist one.

The Text is :
{text}
"""

In [21]:
pre_deepseek = pd.read_csv("deepseek_r1.txt",sep="\t")

In [22]:
pre_deepseek

,sample_name,cell_line,treat,treat_con,treat_time
0,GSM1519353,HCC1419,untreated,Not applicable,Not applicable
1,GSM1519354,HCC1419,untreated,Not applicable,Not applicable
2,GSM1519355,HCC1419,Lapatinib,1μM,9 days
3,GSM1519356,HCC1419,Lapatinib,1μM,9 days
4,GSM1519357,HCC1419,Lapatinib,1μM,indefinitely
...,...,...,...,...,...
5033,GSM9145433,NALM-6,DMSO,Not specified,Not specified
5034,GSM9145434,NALM-6,DMSO,Not specified,Not specified
5035,GSM9145435,NALM-6,albendazole,[not specified],[not specified]
5036,GSM9145436,NALM-6,albendazole,NaN,NaN


In [30]:
from tqdm import tqdm
for i in tqdm(range(len(pre_deepseek))):
    with open('GSM_text/'+pre_deepseek.sample_name[i]+".txt", 'r', encoding='utf-8') as file:
        text = file.read()  # 读取整个文件内容
        
    response = client.chat.completions.create(
        model="alibayram/Qwen3-30B-A3B-Instruct-2507:latest",
        messages=[
            {"role": "system", "content": "You are an expert in biology and need to judge the correctness of information provided based on text."},
            {"role": "user", "content": prompts.format(text = text, cell = pre_deepseek.cell_line[i], 
                                                       drug = str(pre_deepseek.treat[i]), drug_con = str(pre_deepseek.treat_con[i]),
                                                       drug_time = str(pre_deepseek.treat_time[i]))},
        ],
        stream=False)
    with open('GSM_deepseek_correct/'+pre_deepseek.sample_name[i]+"_out.txt", 'w', encoding='utf-8') as f:
        print(response.choices[0].message.content, file=f)

100%|██████████| 5038/5038 [1:02:35<00:00,  1.34it/s]
